HOMEWORK 3

In [57]:
# Dependencies

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

Getting the data

In [58]:
original_df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")
original_df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


Data preparation

In [59]:
# Missing values before imputation
original_df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [60]:
df = original_df.copy()

In [61]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


For missing values:

- Categorical features, replace them with 'NA'
- Numerical features, replace with 0.0

In [62]:
df[df.select_dtypes(include='object').columns] = df.select_dtypes(include='object').fillna('NA')

In [63]:
df[df.select_dtypes(include=np.number).columns] = df.select_dtypes(include=np.number).fillna(0.0)

In [64]:
# Missing values after imputation
df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


**Q1. What is the most frequent observation (mode) for the column industry?**

Answer: Technology

In [65]:
df['industry'].mode()

,industry
0,technology


**Q2. From correlation matrix: what are the two features that have the biggest correlation?**

Answer: interaction_count and lead_score

In [66]:
numerical = df.drop(columns=['converted']).select_dtypes(include=[np.number])


In [67]:
corr = numerical.corr()

# Pairs of variables as given in homework
pairs = {
    "interaction_count and lead_score": corr.loc["interaction_count", "lead_score"],
    "number_of_courses_viewed and lead_score": corr.loc["number_of_courses_viewed", "lead_score"],
    "number_of_courses_viewed and interaction_count": corr.loc["number_of_courses_viewed", "interaction_count"],
    "annual_income and interaction_count": corr.loc["annual_income", "interaction_count"]
}

pairs

{'interaction_count and lead_score': np.float64(0.9157457980418697),
 'number_of_courses_viewed and lead_score': np.float64(0.7572042392104886),
 'number_of_courses_viewed and interaction_count': np.float64(0.7216090038937035),
 'annual_income and interaction_count': np.float64(0.12284235135955959)}

In [68]:
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


Splittin data (using homework code)

In [69]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']


In [70]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

**Q3. Which of these variables has the biggest mutual information score: industry, location, lead_source, employment_status?**

Anwer: lead_source

In [71]:
categorical = df[df.select_dtypes(include='object').columns]

In [72]:
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

df_full_train[categorical.columns].apply(mutual_info_converted_score).round(2).sort_values(ascending=False)

,0
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


**Q4. After logistic regression, what accuracy did you get?**

Answer: 0.65

One-hot encoding categorical variables

In [73]:
train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')

X_val = dv.transform(val_dicts)

Training regression

In [74]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [79]:
model.predict(X_train)

array([1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,

In [80]:
y_pred = model.predict_proba(X_val)[:,1] # Predictions of converted (second column of the array) - Validation dataset

In [81]:
converted_decision = (y_pred >= 0.5)

(y_val == converted_decision).mean().round(2)

np.float64(0.64)

**Q5. Which of following feature has the smallest difference: 'lead_source', 'number_of_courses_viewed', 'interaction_count'?**

Answer: number_of_courses_viewed

In [83]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
original_acc = (y_pred == y_val).mean()

diffs = {}

for col in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:

    train_dicts_temp = df_train.drop(columns=[col]).to_dict(orient='records')
    val_dicts_temp = df_val.drop(columns=[col]).to_dict(orient='records')

    X_train_temp = dv.fit_transform(train_dicts_temp)
    X_val_temp = dv.transform(val_dicts_temp)

    model_temp = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_temp.fit(X_train_temp, y_train)
    y_pred_temp = model_temp.predict(X_val_temp)

    acc_temp = (y_pred_temp == y_val).mean()
    diffs[col] = original_acc - acc_temp

diffs


{'lead_source': np.float64(0.0030000000000000027),
 'number_of_courses_viewed': np.float64(0.0020000000000000018),
 'interaction_count': np.float64(0.04400000000000004)}

**Q6. After regularized logistic regression, which value of C leads to the best accuracy on the validation set?**

Answer: C = 0.001

In [84]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
results = {}

for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = (y_pred == y_val).mean()
    results[C] = round(acc, 3)

results


{1e-06: np.float64(0.598),
 1e-05: np.float64(0.598),
 0.0001: np.float64(0.613),
 0.001: np.float64(0.645)}